# Rear-side POA irradiance in PV single-axis tracking (SAT) systems

Rear-side irradiance is highly nonuniform, so the placement of rear-side irradiance sensors is important. In this exercise, you will analyze how a sensor's location affects its readings and determine the location that best represents the overall rear-side irradiance of a PV array.

Essentially, you will validate the findings of this publication:

[Riedel-Lyngskær N., Lysgaard Andersen N., 2025, Rear irradiance monitoring on tracked bifacial photovoltaic systems, Solar Energy](https://www.sciencedirect.com/science/article/pii/S0038092X25004426?via%3Dihub)


<p align="center">
  <img src="https://raw.githubusercontent.com/AdamRJensen/solar-resource-course/refs/heads/main/figures/rear-poa_sensor_placement.png" width="500">
  <br>
  <em>Optimal position of rear-POA sensors</em>
</p>

In [ ]:
# Install pvlib on Google Colab, as this is not a standard package.
!pip install pvlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 33.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pvlib

## Step 1: Import dataset

This exercise will use the same dataset as in Exercise 1. To simulate the irradiance on the rear side of a PV array, all we need are the base irradiance components (GHI, DNI, DHI). All measurements are in W/m$^2$.

In [ ]:
file_url = 'https://github.com/AdamRJensen/solar-resource-course/raw/refs/heads/main/data/poa_data_for_sat_pv_system.csv'

data = pd.read_csv(file_url, index_col=[0], parse_dates=[0])

data = data[["ghi", "dni", "dhi"]]  # keep only these columns

data = data.loc["2025-07-01", :]  # keep just this example clear-sky day

data.head()

## Step 2: Calculate the tracker rotation angles

First, compute the solar positions for each time interval:

In [ ]:
# Write your code here
solar_position = pvlib.solarposition.get_solarposition(
    time=data.index,
    latitude=55.3221,
    longitude=11.3894
)

Next, simulate the positions of the park's solar trackers.  Typical PV parks use single-axis trackers (i.e., trackers that rotate around one axis instead of two), oriented with their axis aligned horizontally north-south so that the modules can rotate from east in the morning to flat at solar noon to west in the evening.

We can do this using pvlib's [singleaxis](https://pvlib-python.readthedocs.io/en/stable/reference/generated/pvlib.tracking.singleaxis.html) function, which needs sun position information and some basic system configuration information.  One key piece of information is the ground coverage ratio (GCR), which represents the density of PV on the land.  For arrays with small inter-row spacing, the GCR is large.

We will configure the system to backtrack, which means it will rotate away from the sun to prevent row-to-row shading when the sun is low in the sky.

In [ ]:
# Fill out the missing fields

module_height = 2  # [m]
pitch = 5  # [m], space between two rows
gcr =

tracker_position = pvlib.tracking.singleaxis(
    apparent_zenith= ,
    solar_azimuth= ,
    backtrack=True,
    gcr=gcr
)



Here is how the tracker position time series looks. A position of zero means the PV panels are horizontal. In this case, positive positions indicate panels tilted towards the east and negative positions indicate panels tilted towards the west.

You can see the backtracking in the morning: the panels must lay flat to prevent row-to-row shading. As the sun gets higher in the sky, the panels can start to tilt more towards the east. Around 6:00 the sun is high enough for shading to not be a concern, so now the trackers follow the sun normally until around 17:00, when backtracking begins again.

You can set `backtrack=False` in the function above to see how the tracker angle would be if the PV system was not backtracking.

In [ ]:
tracker_position["tracker_theta"].plot(ylabel="Tracker rotation angle [degrees]")

Here are what the corresponding PV module tilt and azimuth values look like. Can you explain why the look the way they do?

In [ ]:
tracker_position[["surface_tilt", "surface_azimuth"]].plot(ylabel="Angle [degrees]")

## Step 3: Simulate rear-side irradiance

Now it's time to run some irradiance simulations. There are many ways of simulating the irradiance on the rear-side of a PV array. Here we will use the [ANTS-2D model](https://pvlib-python.readthedocs.io/en/v0.16.0/reference/generated/pvlib.bifacial.ants2d.get_irradiance.html) since it provides the detail we need while still being easy to use.

To run the model, we need the following information:

- PV tracker position
- solar position
- system geometry parameters
- base irradiance components

Since we're trying to answer the question "which sensor location best approximates the average rear-side irradiance", let's first simulate the average rear-side irradiance:

In [ ]:
# Fill out the missing fields

albedo = 0.2  # [unitless] - typical value for vegetation

# Calculate the average rear-side POA for the entire row
rpoa_average = pvlib.bifacial.ants2d.get_irradiance(
    tracker_rotation= ,
    axis_azimuth=0,  # north-south orientation
    solar_zenith= ,
    solar_azimuth= ,
    height=module_height,
    pitch=pitch,
    gcr=gcr,
    ghi=data["ghi"],
    dni=data["dni"],
    dhi=data["dhi"],
    albedo=albedo,
)

rpoa_average = rpoa_average["poa_back"]

Now we can run the model again, but using the row_segments parameter to specify a sequence of potential sensor locations.

In [ ]:
# Fill out the missing fields

# This input specifies the left/right sides of a sequence of sensor locations.
# The numbers are unitless, representing fractions of the module height.
# Don't worry too much about this input :)
sensor_locations = [(0, 0.01), (0.095, 0.105), (0.195, 0.205), (0.295, 0.305),
                    (0.395, 0.405), (0.495, 0.505), (0.595, 0.605), (0.695, 0.705),
                    (0.795, 0.805), (0.895, 0.905), (0.99, 1.0)]

# Calculate the rear-side POA for each of these segments
rpoa_sensors = pvlib.bifacial.ants2d.get_irradiance(
    tracker_rotation= ,
    axis_azimuth=0,  # north-south orientation
    solar_zenith= ,
    solar_azimuth= ,
    height=module_height,
    pitch=pitch,
    gcr=gcr,
    ghi=data["ghi"],
    dni=data["dni"],
    dhi=data["dhi"],
    albedo=albedo,
    row_segments=sensor_locations,
)

# Make the rear-side POA results into a new dataframe
rpoa_sensors = rpoa_sensors["poa_back"]
rpoa_sensors = pd.DataFrame(rpoa_sensors.T, index=data.index)

Let's take a look at the simulated irradiance time series. Here the colored lines are the irradiance for the eleven sensors we simulated and the average irradiance across the module is shown in a black dashed line.

What do you notice about morning/afternoon symmetry? Can you already tell something about which sensor placement will most accurately approximate the average?

In [ ]:
rpoa_sensors.plot(cmap="tab20")
rpoa_average.plot(c='k', ls='--')

Let's make it a bit more quantitative and compute statistics: root-mean-squared error (RMSE) and mean bias error (MBE).

In [ ]:
diff = rpoa_sensors.subtract(rpoa_average, axis=0)
rmse = (diff**2).mean(axis=0)**0.5
mbe = diff.mean()

rmse.plot.bar(xlabel="Sensor location", ylabel="Root-mean-squared error [Wm⁻²]")

In [ ]:
mbe.plot.bar(xlabel="Sensor location", ylabel="Mean bias error [Wm⁻²]")

Is the sensor location with the lowest RMSE the same as the location with the smallest bias? Can you think of applications where you might care more about RMSE than MBE, and vice versa?

How might these results chance if you calculate error considering only the morning or only the afternoon?

What might change about the results if we used an overcast day instead of a clear-sky day?